# Financial Document RAG: QLoRA Qwen2.5-1.5B + BGE + FAISS + Reranker over SEC 10-K Filings

**Runtime:** Google Colab, GPU = **T4**. Run cells top to bottom.

**What is fixed / improved vs the old notebook**
- Pinned `transformers<5` (v5 removed `warmup_ratio` and broke training). Uses `warmup_steps`, no hiding `try/except`.
- **fp16 on T4** (T4 has no real bf16).
- **Train/inference mismatch fixed:** training mixes (a) gold-only context, (b) gold + distractor chunks, (c) **unanswerable** questions with a refusal answer.
- Evaluation under 3 conditions (gold / noisy / unanswerable), base vs fine-tuned.
- Retrieval: dense BGE + FAISS, plus **BGE reranker**, with recall@k before/after.
- Live filings from **free SEC EDGAR** (no paid sec-api key).

Put on your resume only the numbers this notebook prints.

## 1. Setup

In [ ]:
!pip install --no-cache-dir --force-reinstall "transformers==4.57.1" "tokenizers>=0.22,<=0.23" "huggingface_hub>=0.34,<1.0"
!pip install -q peft accelerate bitsandbytes datasets sentence-transformers faiss-cpu langchain-text-splitters beautifulsoup4

In [ ]:
import transformers
from transformers import PreTrainedModel, TrainingArguments
print(transformers.__version__)

**Restart the runtime now** (Runtime > Restart session) so the pinned transformers loads. After the restart, **skip the install cell** and continue from the next one.

In [ ]:
import re, string, random, gc, json, time, contextlib
from collections import Counter
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import transformers

from datasets import load_dataset, Dataset
from sklearn.model_selection import GroupShuffleSplit
from transformers import (AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
                          TrainingArguments, Trainer, DataCollatorForSeq2Seq)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

assert int(transformers.__version__.split(".")[0]) < 5, \
    f"transformers {transformers.__version__} is too new: run the install cell, then restart the runtime."
assert torch.cuda.is_available(), "Turn on a GPU: Runtime > Change runtime type > T4 GPU"

MODEL_NAME   = "Qwen/Qwen2.5-1.5B-Instruct"
DATASET_NAME = "virattt/financial-qa-10K"
MAX_LEN = 1024
SEED = 42
QUICK_TEST = False        # True = train on 600 examples only, to check the pipeline in ~5 min
N_EVAL = 120              # examples per evaluation condition

compute_dtype = torch.float16     # T4 = fp16 (bf16 is only emulated on T4 and very slow)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("transformers", transformers.__version__, "| GPU:", torch.cuda.get_device_name(0))

## 2. Load, clean and split the data (by ticker)
Many questions repeat the same fact, so we hold out whole **companies** to avoid leakage.

In [ ]:
raw = load_dataset(DATASET_NAME, split="train")
df = raw.to_pandas()
print("Shape:", df.shape, "| tickers:", df["ticker"].nunique())

for col in ["question", "answer", "context"]:
    df[col] = df[col].fillna("").str.strip()
before = len(df)
df = df[(df.question != "") & (df.answer != "") & (df.context != "")]
df = df.drop_duplicates(subset=["question", "answer", "context"]).reset_index(drop=True)
print(f"Rows: {before} -> {len(df)}")

gss = GroupShuffleSplit(n_splits=1, test_size=0.1, random_state=SEED)
train_idx, test_idx = next(gss.split(df, groups=df["ticker"]))
train_df = df.iloc[train_idx].reset_index(drop=True)
test_df  = df.iloc[test_idx].reset_index(drop=True)

assert set(train_df.ticker).isdisjoint(set(test_df.ticker))
print("Train rows:", len(train_df), "| tickers:", train_df.ticker.nunique())
print("Test rows :", len(test_df),  "| tickers:", test_df.ticker.nunique(), sorted(test_df.ticker.unique()))

## 3. Prompt format and training-data construction
Three kinds of training example (so the model sees what RAG will really feed it):

| kind | share | context | target |
|---|---|---|---|
| gold | 55% | the one correct passage | gold answer |
| noisy | 30% | gold + 2 distractor passages (same company), shuffled | gold answer |
| unanswerable | 15% | passages from a **different** company | refusal sentence |

Loss is computed only on answer tokens.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

REFUSAL = "I cannot find the answer in the provided filing context."
SYSTEM_PROMPT = ("You are a financial analyst. Answer the question using only the numbered context passages "
                 "from SEC 10-K filings. Be concise and include specific numbers when available. "
                 f"If the context does not contain the answer, say exactly: {REFUSAL}")

def format_contexts(ctxs):
    return "\n\n".join(f"[{i+1}] {c}" for i, c in enumerate(ctxs))

def build_prompt(question, contexts):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": f"Question:\n{question}\n\nContext:\n{format_contexts(contexts)}"},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def make_pools(frame):
    by_ticker = frame.groupby("ticker")["context"].apply(lambda s: sorted(set(s))).to_dict()
    pool = [(t, c) for t, cs in by_ticker.items() for c in cs]
    return by_ticker, pool

def build_example(row, rng, mode, by_ticker, pool):
    gold, ticker = row["context"], row["ticker"]
    if mode == "gold":
        ctxs, ans = [gold], row["answer"]
    elif mode == "noisy":
        same = [c for c in by_ticker.get(ticker, []) if c != gold]
        if len(same) < 2:
            same += [c for t, c in rng.sample(pool, min(10, len(pool))) if c != gold]
        distract = rng.sample(same, min(2, len(same)))
        ctxs = [c[:1200] for c in distract] + [gold]
        rng.shuffle(ctxs)
        ans = row["answer"]
    else:  # "unans": passages from other companies, answer = refusal
        others = [c for t, c in rng.sample(pool, min(60, len(pool))) if t != ticker]
        ctxs = [c[:1200] for c in others[:rng.choice([1, 2])]]
        ans = REFUSAL
    return {"question": row["question"], "contexts": ctxs, "answer": ans,
            "gold_answer": row["answer"], "ticker": ticker, "kind": mode}

def sample_mode(rng):
    return rng.choices(["gold", "noisy", "unans"], weights=[0.55, 0.30, 0.15])[0]

def tokenize_example(ex):
    prompt_ids = tokenizer(build_prompt(ex["question"], ex["contexts"]), add_special_tokens=False)["input_ids"]
    answer_ids = tokenizer(ex["answer"] + "<|im_end|>", add_special_tokens=False)["input_ids"]
    ids = prompt_ids + answer_ids
    return {"input_ids": ids, "attention_mask": [1] * len(ids),
            "labels": [-100] * len(prompt_ids) + answer_ids}

def make_dataset(examples):
    toks = [tokenize_example(e) for e in examples]
    kept = [t for t in toks if len(t["input_ids"]) <= MAX_LEN]
    print(f"Kept {len(kept)} / {len(toks)} examples (dropped {len(toks) - len(kept)} longer than {MAX_LEN} tokens)")
    return Dataset.from_list(kept)

rng = random.Random(SEED)
tr_by_ticker, tr_pool = make_pools(train_df)
te_by_ticker, te_pool = make_pools(test_df)

train_rows = train_df.to_dict("records")
if QUICK_TEST:
    train_rows = train_rows[:600]
train_ex = [build_example(r, rng, sample_mode(rng), tr_by_ticker, tr_pool) for r in train_rows]
print("Training kinds:", Counter(e["kind"] for e in train_ex))

val_rows = test_df.sample(min(150, len(test_df)), random_state=SEED).to_dict("records")
val_ex = [build_example(r, rng, sample_mode(rng), te_by_ticker, te_pool) for r in val_rows]

train_ds = make_dataset(train_ex)
val_ds   = make_dataset(val_ex)
print("Mean token length:", int(np.mean([len(x) for x in train_ds["input_ids"]])),
      "| max:", max(len(x) for x in train_ds["input_ids"]))

# Look at one unanswerable example
ex = next(e for e in train_ex if e["kind"] == "unans")
print("\n" + build_prompt(ex["question"], ex["contexts"]) + ex["answer"] + "<|im_end|>")

## 4. Load Qwen in 4-bit and add LoRA (QLoRA)
NF4 4-bit base model + double quantization; LoRA `r=16, alpha=32` on all attention and MLP projections.

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map={"": 0}, torch_dtype=compute_dtype,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)   # gradient checkpointing, fp32 norms

lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 5. Train
2 epochs, effective batch 16, lr 2e-4, cosine schedule. Roughly 60 to 90 minutes on a T4 (use `QUICK_TEST=True` first to check everything runs).

In [ ]:
# Make gradients flow through frozen embeddings + use non-reentrant checkpointing
model.enable_input_require_grads()
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.config.use_cache = False
model.train()

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Trainable params:", trainable)      # should be ~18.4 million, NOT 0

args = TrainingArguments(
    output_dir="outputs",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=2,
    learning_rate=2e-4,
    warmup_steps=25,
    lr_scheduler_type="cosine",
    weight_decay=0.01,
    optim="paged_adamw_8bit",
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="no",
    fp16=True,
    bf16=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    remove_unused_columns=False,
    report_to="none",
    seed=SEED,
)
collator = DataCollatorForSeq2Seq(tokenizer, padding=True, label_pad_token_id=-100, pad_to_multiple_of=8)

trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds, data_collator=collator)
train_output = trainer.train()
print(train_output.metrics)

In [ ]:
log_df = pd.DataFrame(trainer.state.log_history)
plt.figure(figsize=(8, 4))
tl = log_df.dropna(subset=["loss"])
plt.plot(tl["step"], tl["loss"], label="train loss")
if "eval_loss" in log_df.columns:
    ev = log_df.dropna(subset=["eval_loss"])
    plt.plot(ev["step"], ev["eval_loss"], marker="o", label="eval loss (held-out tickers)")
plt.xlabel("step"); plt.ylabel("loss"); plt.legend(); plt.grid(alpha=0.3)
plt.title("QLoRA fine-tuning of Qwen2.5-1.5B"); plt.show()

SAVE_DIR = "qwen25_1p5b_fin_qlora"
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print("Adapter saved to", SAVE_DIR)
# Download it (optional): !zip -r adapter.zip qwen25_1p5b_fin_qlora
del trainer
gc.collect(); torch.cuda.empty_cache()

## 6. Evaluate: base vs fine-tuned, three conditions
Held-out companies only. `model.disable_adapter()` = base model.

- **gold:** only the correct passage (easy)
- **noisy:** correct passage + 2 distractors (closer to real RAG)
- **unanswerable:** passages from another company; the right behaviour is to refuse

Metrics: token F1, exact match, refusal rate (on answerable sets this is a *false* refusal; on unanswerable it is the *correct* refusal).

In [ ]:
@torch.no_grad()
def generate(questions, contexts_list, adapter=True, batch_size=8, max_new_tokens=100):
    model.eval()
    model.config.use_cache = True
    tokenizer.padding_side = "left"
    prompts = [build_prompt(q, c) for q, c in zip(questions, contexts_list)]
    outputs = []
    for i in range(0, len(prompts), batch_size):
        enc = tokenizer(prompts[i:i + batch_size], return_tensors="pt", padding=True,
                        add_special_tokens=False).to(model.device)
        ctx = contextlib.nullcontext() if adapter else model.disable_adapter()
        with ctx:
            out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                                 pad_token_id=tokenizer.pad_token_id)
        new = out[:, enc["input_ids"].shape[1]:]
        outputs += [t.strip() for t in tokenizer.batch_decode(new, skip_special_tokens=True)]
    tokenizer.padding_side = "right"
    return outputs

def normalize(text):
    text = text.lower()
    text = "".join(ch for ch in text if ch not in set(string.punctuation))
    return " ".join(text.split())

def token_f1(pred, gold):
    p, g = normalize(pred).split(), normalize(gold).split()
    if not p or not g:
        return float(p == g)
    overlap = sum((Counter(p) & Counter(g)).values())
    if overlap == 0:
        return 0.0
    prec, rec = overlap / len(p), overlap / len(g)
    return 2 * prec * rec / (prec + rec)

def exact_match(pred, gold):
    return float(normalize(pred) == normalize(gold))

def is_refusal(text):
    t = text.lower()
    return any(s in t for s in ["cannot find", "can't find", "not contain", "does not provide", "no information"])

In [ ]:
eval_rows = test_df.sample(min(N_EVAL, len(test_df)), random_state=SEED).to_dict("records")
rng_eval = random.Random(SEED + 1)
eval_sets = {m: [build_example(r, rng_eval, m, te_by_ticker, te_pool) for r in eval_rows]
             for m in ["gold", "noisy", "unans"]}

records, all_preds = [], {}
for mode, exs in eval_sets.items():
    qs = [e["question"] for e in exs]; cs = [e["contexts"] for e in exs]
    for name, adapter in [("Base Qwen2.5-1.5B", False), ("QLoRA fine-tuned", True)]:
        preds = generate(qs, cs, adapter=adapter)
        all_preds[(mode, name)] = preds
        ref = np.mean([is_refusal(p) for p in preds])
        if mode == "unans":
            records.append({"condition": mode, "model": name, "F1": np.nan, "Exact match": np.nan,
                            "Refusal rate": ref, "Avg words": np.mean([len(p.split()) for p in preds])})
        else:
            golds = [e["answer"] for e in exs]
            records.append({"condition": mode, "model": name,
                            "F1": np.mean([token_f1(p, g) for p, g in zip(preds, golds)]),
                            "Exact match": np.mean([exact_match(p, g) for p, g in zip(preds, golds)]),
                            "Refusal rate": ref, "Avg words": np.mean([len(p.split()) for p in preds])})

results = pd.DataFrame(records).set_index(["condition", "model"]).round(3)
results.to_csv("generation_eval_results.csv")
results

In [ ]:
pd.set_option("display.max_colwidth", 160)
exs = eval_sets["noisy"]
pd.DataFrame({"ticker": [e["ticker"] for e in exs], "question": [e["question"] for e in exs],
              "gold": [e["answer"] for e in exs],
              "base": all_preds[("noisy", "Base Qwen2.5-1.5B")],
              "fine_tuned": all_preds[("noisy", "QLoRA fine-tuned")]}).head(8)

## 7. Evaluate retrieval: BGE + FAISS, then + reranker
For each held-out question: can we find its gold passage among all held-out passages? This pool is small, so treat it as a sanity check, not a headline number.
BGE vectors are normalised, so inner product = cosine similarity (`IndexFlatIP`). The reranker (cross-encoder) re-scores the top 20 dense hits.

In [ ]:
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder

embedder = SentenceTransformer("BAAI/bge-base-en-v1.5", device="cuda")
reranker = CrossEncoder("BAAI/bge-reranker-base", device="cuda", max_length=512)
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "

def embed(texts, is_query=False):
    if is_query:
        texts = [QUERY_PREFIX + t for t in texts]
    return embedder.encode(texts, normalize_embeddings=True, batch_size=32,
                           show_progress_bar=False, convert_to_numpy=True).astype("float32")

unique_ctx = test_df["context"].unique().tolist()
ctx_id = {c: i for i, c in enumerate(unique_ctx)}
eval_index = faiss.IndexFlatIP(embed(unique_ctx[:2]).shape[1])
eval_index.add(embed(unique_ctx))

questions = test_df["question"].tolist()
gold_ids = test_df["context"].map(ctx_id).values
_, top_ids = eval_index.search(embed(questions, is_query=True), 20)

def recall_at(ranked, k):
    return np.mean([gold_ids[i] in ranked[i][:k] for i in range(len(gold_ids))])

dense = {k: recall_at(top_ids, k) for k in [1, 3, 5]}

reranked = []
for qi, q in enumerate(questions):
    cand = top_ids[qi]
    scores = reranker.predict([(q, unique_ctx[c]) for c in cand], batch_size=32, show_progress_bar=False)
    reranked.append([cand[j] for j in np.argsort(-scores)])

rerank = {k: recall_at(reranked, k) for k in [1, 3, 5]}
retrieval_results = pd.DataFrame({"Dense (BGE+FAISS)": dense, "Dense + reranker": rerank}).T
retrieval_results.columns = [f"Recall@{k}" for k in retrieval_results.columns]
print(f"{len(unique_ctx)} candidate passages, {len(questions)} questions")
retrieval_results.round(3)

## 8. Full RAG on a live 10-K (free SEC EDGAR, no paid key)
`question -> BGE top-20 (FAISS) -> reranker top-3 -> numbered context -> fine-tuned Qwen -> answer + sources`

SEC requires a descriptive `User-Agent` with your name and email. Set it below.

In [ ]:
import requests
from bs4 import BeautifulSoup
from langchain_text_splitters import RecursiveCharacterTextSplitter

SEC_USER_AGENT = "Your Name your.email@example.com"  # Replace with your contact before SEC requests
TICKER = "AAPL"
assert "example.com" not in SEC_USER_AGENT, "Put your real name/email in SEC_USER_AGENT (SEC requires it)."
HEADERS = {"User-Agent": SEC_USER_AGENT}

def get_latest_10k_url(ticker):
    tickers = requests.get("https://www.sec.gov/files/company_tickers.json", headers=HEADERS, timeout=30).json()
    cik = next(v["cik_str"] for v in tickers.values() if v["ticker"].upper() == ticker.upper())
    sub = requests.get(f"https://data.sec.gov/submissions/CIK{int(cik):010d}.json", headers=HEADERS, timeout=30).json()
    r = sub["filings"]["recent"]
    for form, acc, doc, date in zip(r["form"], r["accessionNumber"], r["primaryDocument"], r["filingDate"]):
        if form == "10-K":
            return f"https://www.sec.gov/Archives/edgar/data/{int(cik)}/{acc.replace('-', '')}/{doc}", date
    raise ValueError(f"No 10-K found for {ticker}")

def html_to_text(html):
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(["script", "style", "ix:header"]):
        tag.decompose()
    text = soup.get_text("\n")
    text = re.sub(r"[ \t\xa0]+", " ", text)
    return re.sub(r"\n\s*\n+", "\n\n", text).strip()

def extract_core_sections(text):
    # Keep Item 1 (Business) through Item 7A, i.e. cut before Item 8 (financial statements).
    # Table-of-contents entries match too, so take the start/end pair spanning the most text.
    starts = [m.start() for m in re.finditer(r"item\s*1\.?\s*[\-\u2013\u2014:]?\s*business", text, re.I)]
    ends   = [m.start() for m in re.finditer(r"item\s*8\.?\s*[\-\u2013\u2014:]?\s*financial\s+statements", text, re.I)]
    best = None
    for s in starts:
        e = next((x for x in ends if x > s), None)
        if e and (best is None or e - s > best[1] - best[0]):
            best = (s, e)
    if best and best[1] - best[0] > 20000:
        return text[best[0]:best[1]]
    print("Section markers not found, using the full document.")
    return text

filing_url, filing_date = get_latest_10k_url(TICKER)
print("Filing:", filing_url, "| filed", filing_date)
html = requests.get(filing_url, headers=HEADERS, timeout=60).text
filing_text = extract_core_sections(html_to_text(html))
print("Characters:", len(filing_text))

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
chunks = splitter.split_text(filing_text)
vecs = embed(chunks)
rag_index = faiss.IndexFlatIP(vecs.shape[1])
rag_index.add(vecs)
print("Chunks indexed:", len(chunks))

In [ ]:
def retrieve_context(query, k_dense=20, k_final=3):
    _, ids = rag_index.search(embed([query], is_query=True), k_dense)
    cand = [int(i) for i in ids[0]]
    scores = reranker.predict([(query, chunks[i]) for i in cand], batch_size=32, show_progress_bar=False)
    order = np.argsort(-scores)[:k_final]
    return [chunks[cand[j]] for j in order]

def rag_answer(question, adapter=True):
    ctxs = retrieve_context(question)
    return generate([question], [ctxs], adapter=adapter)[0], ctxs

demo_questions = [
    "What are the main risk factors the company highlights?",
    "How did total net sales change compared to the previous year?",
    "What drove the change in gross margin?",
    "What is the name of the CEO's pet dog?",     # unanswerable on purpose: should refuse
]
for q in demo_questions:
    ft, ctxs = rag_answer(q, adapter=True)
    base, _ = rag_answer(q, adapter=False)
    print("Q:", q)
    print("  Fine-tuned + RAG:", ft)
    print("  Base + RAG      :", base)
    print("  Top source      :", ctxs[0][:200].replace("\n", " "), "...")
    print("-" * 90)

In [ ]:
# Optional interactive loop (type x to stop)
while True:
    q = input(f"Ask about {TICKER}'s 10-K (x to quit): ")
    if q.strip().lower() == "x":
        break
    ans, ctxs = rag_answer(q)
    print("\nAnswer:", ans)
    for i, c in enumerate(ctxs, 1):
        print(f"\n[{i}] {c[:300].replace(chr(10), ' ')} ...")
    print("-" * 90)

## 9. Study notes and resume

**Pipeline:** SEC EDGAR 10-K ingestion -> section extraction -> chunking (1000/150) -> BGE embeddings -> FAISS -> cross-encoder rerank -> numbered context -> QLoRA Qwen2.5-1.5B -> grounded answer with sources.

**Design choices**
- **QLoRA:** 4-bit NF4 frozen base (fits a T4), small LoRA adapters trained (about 1.2% of params).
- **Loss only on answer tokens:** learns answering behaviour, not prompt memorisation.
- **Split by ticker:** the dataset has near-duplicate questions per company; a random split inflates scores.
- **Noisy + unanswerable training mix:** matches what retrieval really returns and teaches refusal instead of hallucination.
- **Reranker:** dense retrieval is fast but coarse; the cross-encoder improves precision at the top.
- **Why fine-tune if RAG gives context?** RAG supplies *facts*; fine-tuning teaches *behaviour* (concise, grounded, refuses when unsupported). It adds no financial knowledge.

**Honest limitations**
- Training data: FY2023 only, 69 companies, short extractive answers.
- Tables in 10-Ks are flattened to text (weak spot).
- Token F1 is crude; spot-check ~50 answers by hand or with an LLM judge.
- Retrieval recall is measured on a small pool; real filings are harder.

**Resume bullets (fill in brackets from YOUR printed results, never invent numbers)**
> - Built an end-to-end RAG pipeline over SEC 10-K filings: EDGAR ingestion, chunking, BGE embeddings + FAISS retrieval, cross-encoder reranking, and a QLoRA-fine-tuned Qwen2.5-1.5B generator.
> - Fine-tuned on ~6.2k financial QA examples (split by company, with distractor and unanswerable cases); on 7 unseen companies, F1 rose from [base] to [tuned] with noisy context and correct-refusal rate from [x]% to [y]%.
> - Reranking lifted Recall@1 from [a] to [b] on held-out filings.